# 03 — EDA & Behavioural Analysis
**Goal**: Move beyond summary statistics. Find the *behavioural patterns* that separate churners from loyalists — flight trajectories, points hoarding, seasonal disengagement, and CLV distortions. Every chart should answer a business question.

**Input**: `data/processed/02_cleaned.csv` + `data/processed/02_activity_clean.csv` + `data/processed/02_churn_labels.csv`  
**Output**: Figures saved to `reports/figures/` — these go directly into the final report.

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

PROC = Path('../data/processed')
FIG  = Path('../reports/figures')
os.makedirs(FIG, exist_ok=True)

KEY          = 'loyalty_number'
YEAR_COL     = 'year'
MONTH_COL    = 'month'
FLIGHTS_COL  = 'total_flights'
POINTS_ACC   = 'points_accumulated'
POINTS_RED   = 'points_redeemed'
CLV_COL      = 'clv'
CARD_COL     = 'loyalty_card'

CHURN_COLOR  = '#C0392B'
RETAIN_COLOR = '#2471A3'
NEUTRAL_COLOR = '#566573'

PREDICTION_CUTOFF = pd.Timestamp('2016-12-01')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

---
## 1. Load Data

In [ ]:
members  = pd.read_csv(PROC / '02_cleaned.csv')
activity = pd.read_csv(PROC / '02_activity_clean.csv')
labels   = pd.read_csv(PROC / '02_churn_labels.csv')

activity['period'] = pd.to_datetime(
    activity[YEAR_COL].astype(str) + '-' +
    activity[MONTH_COL].astype(str).str.zfill(2)
)
activity = activity.merge(labels, on=KEY, how='inner')

# Resolve actual column names for points
for cand in ['total_points_accumulated', 'points_accumulated', POINTS_ACC]:
    if cand in members.columns: pts_acc = cand; break
for cand in ['total_points_redeemed', 'points_redeemed', POINTS_RED]:
    if cand in members.columns: pts_red = cand; break
flights_col_m = 'total_flights_historical' if 'total_flights_historical' in members.columns else FLIGHTS_COL

print(f'members  : {members.shape}   churn rate: {members["churned"].mean():.1%}')
print(f'activity : {activity.shape}')

---
## 2. Flight Trajectory — Do Churners Decline Gradually or Stop Abruptly?
The answer determines whether early-warning intervention is possible.

In [ ]:
lookback = activity[
    (activity['period'] <= PREDICTION_CUTOFF) &
    (activity['period'] >= PREDICTION_CUTOFF - pd.DateOffset(months=11))
].copy()

lookback['rel_month'] = (
    (lookback['period'].dt.year  - PREDICTION_CUTOFF.year)  * 12 +
    (lookback['period'].dt.month - PREDICTION_CUTOFF.month)
)

trajectory = (
    lookback.groupby(['rel_month', 'churned'])[FLIGHTS_COL]
    .mean().reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = trajectory[trajectory['churned']==churn_val]
    ax.plot(sub['rel_month'], sub[FLIGHTS_COL], color=color,
            linewidth=2.2, label=label, marker='o', markersize=4)
    ax.fill_between(sub['rel_month'], sub[FLIGHTS_COL], alpha=0.08, color=color)

ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Prediction cutoff')
ax.set_xlabel('Months before prediction cutoff')
ax.set_ylabel('Avg flights per member')
ax.set_title('Flight trajectory: churners vs retained (12 months before cutoff)')
ax.set_xticks(range(-11, 1))
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '03_flight_trajectory.png', bbox_inches='tight')
plt.show()

churn_last  = trajectory[(trajectory['churned']==1)&(trajectory['rel_month']==0)][FLIGHTS_COL].values[0]
churn_first = trajectory[(trajectory['churned']==1)&(trajectory['rel_month']==-11)][FLIGHTS_COL].values[0]
print(f'Churner avg flights — 12m ago: {churn_first:.2f}, at cutoff: {churn_last:.2f}')
print(f'Decline: {((churn_last-churn_first)/max(churn_first,0.01))*100:.1f}%')

---
## 3. Points Hoarding — Loss Aversion Signal
Members who accumulate but never redeem points are either deeply loyal or deeply disengaged. The trajectory tells you which.

In [ ]:
members['redemption_ratio'] = (
    members[pts_red] / members[pts_acc].replace(0, np.nan)
).fillna(0).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = members[members['churned']==churn_val]['redemption_ratio']
    axes[0].hist(sub, bins=40, alpha=0.55, color=color, label=label,
                 density=True, edgecolor='white')
axes[0].set_xlabel('Redemption ratio (redeemed / accumulated)')
axes[0].set_ylabel('Density')
axes[0].set_title('Points redemption ratio by churn status')
axes[0].legend()

plot_data = members[[pts_acc, 'churned']].copy()
plot_data['churned_label'] = plot_data['churned'].map({0:'Retained',1:'Churned'})
sns.boxplot(data=plot_data, x='churned_label', y=pts_acc,
            palette={'Retained':RETAIN_COLOR,'Churned':CHURN_COLOR},
            ax=axes[1], width=0.5, fliersize=2)
axes[1].set_xlabel('')
axes[1].set_ylabel('Total points accumulated')
axes[1].set_title('Points accumulated: churners vs retained')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

plt.suptitle('Loss aversion signal — points hoarding behaviour', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '03_points_hoarding.png', bbox_inches='tight')
plt.show()

for churn_val, label in [(0,'Retained'),(1,'Churned')]:
    r = members[members['churned']==churn_val]['redemption_ratio'].median()
    print(f'{label} median redemption ratio: {r:.3f}')

---
## 4. Seasonal Engagement Patterns
Do churners disengage in specific seasons? Seasonal drop-off tells you *when* to send retention nudges.

In [ ]:
seasonal = (
    lookback.groupby([MONTH_COL,'churned'])[FLIGHTS_COL]
    .mean().reset_index()
)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = seasonal[seasonal['churned']==churn_val].sort_values(MONTH_COL)
    ax.plot(sub[MONTH_COL], sub[FLIGHTS_COL], color=color,
            linewidth=2, label=label, marker='s', markersize=5)

ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Month of year')
ax.set_ylabel('Avg flights per member')
ax.set_title('Seasonal flight patterns: churners vs retained')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '03_seasonal_patterns.png', bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal volatility: how much does each member deviate from their own baseline?
member_monthly = (
    lookback.groupby([KEY,MONTH_COL])[FLIGHTS_COL].mean().reset_index()
)
member_avg = (
    lookback.groupby(KEY)[FLIGHTS_COL].mean().reset_index()
    .rename(columns={FLIGHTS_COL:'member_avg'})
)
member_monthly = member_monthly.merge(member_avg, on=KEY, how='left')
member_monthly['deviation'] = abs(member_monthly[FLIGHTS_COL] - member_monthly['member_avg'])

seasonal_vol = (
    member_monthly.groupby(KEY)['deviation'].mean().reset_index()
    .rename(columns={'deviation':'seasonal_volatility'})
)
seasonal_vol = seasonal_vol.merge(labels, on=KEY, how='inner')

fig, ax = plt.subplots(figsize=(10, 4))
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = seasonal_vol[seasonal_vol['churned']==churn_val]['seasonal_volatility']
    ax.hist(sub, bins=40, alpha=0.55, color=color, label=label,
            density=True, edgecolor='white')
ax.set_xlabel('Seasonal volatility (avg deviation from personal baseline)')
ax.set_ylabel('Density')
ax.set_title('Seasonal volatility by churn status')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '03_seasonal_volatility.png', bbox_inches='tight')
plt.show()

seasonal_vol.to_csv(PROC / '03_seasonal_volatility.csv', index=False)
print('Saved seasonal_volatility feature candidate.')

---
## 5. CLV vs Behaviour — The Core Business Tension
Does existing CLV identify high-risk members? This chart is the centrepiece of your report.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = members[members['churned']==churn_val]
    ax.scatter(sub[flights_col_m], sub[CLV_COL],
               c=color, label=label, alpha=0.25, s=12, linewidths=0)

ax.set_xlabel('Total historical flights')
ax.set_ylabel('CLV ($)')
ax.set_title('CLV vs flight activity — churners scatter through ALL CLV bands')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(FIG / '03_clv_vs_flights_scatter.png', bbox_inches='tight')
plt.show()

clv_75 = members[CLV_COL].quantile(0.75)
high_clv_churn = members[members[CLV_COL]>=clv_75]['churned'].mean()
print(f'Top-25% CLV members churn rate: {high_clv_churn:.1%}')
print('→ CLV alone does NOT identify at-risk members.')

In [ ]:
if CARD_COL in members.columns:
    clv_tier = (
        members.groupby([CARD_COL,'churned'])[CLV_COL].median().reset_index()
    )
    clv_tier['churn_label'] = clv_tier['churned'].map({0:'Retained',1:'Churned'})

    fig, ax = plt.subplots(figsize=(10,5))
    sns.barplot(data=clv_tier, x=CARD_COL, y=CLV_COL, hue='churn_label',
                palette={'Retained':RETAIN_COLOR,'Churned':CHURN_COLOR}, ax=ax)
    ax.set_xlabel('Loyalty card tier')
    ax.set_ylabel('Median CLV ($)')
    ax.set_title('Median CLV by card tier and churn status')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.legend(title='')
    plt.tight_layout()
    plt.savefig(FIG / '03_clv_by_tier_churn.png', bbox_inches='tight')
    plt.show()

---
## 6. Hyperbolic Discounting — Recency vs History
A member with 20 flights in the last 3 months is very different from one who flew 20 times spread over 3 years. This chart visualises the *danger zone*.

In [ ]:
recent_3m = (
    activity[
        (activity['period'] > PREDICTION_CUTOFF - pd.DateOffset(months=3)) &
        (activity['period'] <= PREDICTION_CUTOFF)
    ]
    .groupby(KEY)[FLIGHTS_COL].sum().reset_index()
    .rename(columns={FLIGHTS_COL:'flights_last_3m'})
)
full_12m = (
    lookback.groupby(KEY)[FLIGHTS_COL].sum().reset_index()
    .rename(columns={FLIGHTS_COL:'flights_12m'})
)
recency_df = recent_3m.merge(full_12m, on=KEY).merge(labels, on=KEY)
recency_df['declining'] = (
    (recency_df['flights_12m']     > recency_df['flights_12m'].median()) &
    (recency_df['flights_last_3m'] < recency_df['flights_last_3m'].median())
).astype(int)

declining_churn = recency_df[recency_df['declining']==1]['churned'].mean()
stable_churn    = recency_df[recency_df['declining']==0]['churned'].mean()
print(f'Declining trajectory churn rate : {declining_churn:.1%}')
print(f'Stable trajectory churn rate    : {stable_churn:.1%}')

fig, ax = plt.subplots(figsize=(10, 5))
for churn_val, label, color in [(0,'Retained',RETAIN_COLOR),(1,'Churned',CHURN_COLOR)]:
    sub = recency_df[recency_df['churned']==churn_val]
    ax.scatter(sub['flights_12m'], sub['flights_last_3m'],
               c=color, label=label, alpha=0.2, s=10, linewidths=0)

ax.axvline(recency_df['flights_12m'].median(), color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(recency_df['flights_last_3m'].median(), color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.text(recency_df['flights_12m'].quantile(0.85),
        recency_df['flights_last_3m'].quantile(0.05),
        'High history\nLow recent\n← DANGER ZONE',
        fontsize=9, color=CHURN_COLOR, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FADBD8', alpha=0.8))
ax.set_xlabel('Total flights (12 months)')
ax.set_ylabel('Flights (last 3 months)')
ax.set_title('Hyperbolic discounting: recent vs historical activity')
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(FIG / '03_recency_vs_history.png', bbox_inches='tight')
plt.show()

---
## 7. Tier Stagnation — Status Quo Bias

In [ ]:
if CARD_COL in members.columns:
    tier_churn = (
        members.groupby(CARD_COL)['churned']
        .agg(['mean','count','sum'])
        .rename(columns={'mean':'churn_rate','count':'total','sum':'n_churned'})
        .sort_values('churn_rate', ascending=False).reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors_bar = [CHURN_COLOR if r > tier_churn['churn_rate'].mean()
                  else RETAIN_COLOR for r in tier_churn['churn_rate']]
    bars = axes[0].bar(tier_churn[CARD_COL], tier_churn['churn_rate'],
                       color=colors_bar, alpha=0.85, edgecolor='white')
    axes[0].axhline(tier_churn['churn_rate'].mean(), color='gray',
                    linestyle='--', linewidth=1, label='Overall avg')
    for bar, val in zip(bars, tier_churn['churn_rate']):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                     f'{val:.1%}', ha='center', fontsize=10)
    axes[0].set_title('Churn rate by loyalty tier')
    axes[0].set_ylabel('Churn rate')
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[0].legend()

    axes[1].bar(tier_churn[CARD_COL], tier_churn['total'],
                color=NEUTRAL_COLOR, alpha=0.4, label='Retained', edgecolor='white')
    axes[1].bar(tier_churn[CARD_COL], tier_churn['n_churned'],
                color=CHURN_COLOR, alpha=0.7, label='Churned', edgecolor='white')
    axes[1].set_title('Member count by tier')
    axes[1].set_ylabel('Members')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
    axes[1].legend()

    plt.suptitle('Status quo bias — tier composition and churn rate', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG / '03_tier_stagnation.png', bbox_inches='tight')
    plt.show()
    display(tier_churn)

---
## 8. Demographic Breakdown — Demographics vs Behaviour

In [ ]:
demo_cols_to_check = [
    ('marital_status','Marital status'),
    ('education','Education level'),
    ('gender','Gender'),
    ('province','Province'),
]
available_demos = [(c,t) for c,t in demo_cols_to_check if c in members.columns]

if available_demos:
    n = len(available_demos)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1: axes = [axes]
    for ax, (col, title) in zip(axes, available_demos):
        cbd = (members.groupby(col)['churned'].mean()
               .sort_values(ascending=False).reset_index())
        ax.bar(cbd[col].astype(str), cbd['churned'],
               color='steelblue', alpha=0.8, edgecolor='white')
        ax.axhline(members['churned'].mean(), color='red',
                   linestyle='--', linewidth=1, alpha=0.7, label='Overall avg')
        ax.set_title(title)
        ax.set_ylabel('Churn rate')
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        ax.tick_params(axis='x', rotation=30)
        ax.legend(fontsize=8)
    plt.suptitle('Churn rate by demographic segment', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG / '03_churn_by_demographics.png', bbox_inches='tight')
    plt.show()

---
## 9. Correlation Heatmap — Spot Multicollinearity Before Modelling

In [ ]:
numeric_members = members.select_dtypes(include='number')
drop_cols = [KEY] if KEY in numeric_members.columns else []
corr_df   = numeric_members.drop(columns=drop_cols, errors='ignore')
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('Correlation matrix — numeric features', fontsize=13)
plt.tight_layout()
plt.savefig(FIG / '03_correlation_heatmap.png', bbox_inches='tight')
plt.show()

high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i,j]
        if abs(val) > 0.85:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], round(val,3)))

if high_corr:
    print('High correlation pairs (|r| > 0.85) — consider dropping one in modelling:')
    for c1, c2, v in high_corr:
        print(f'  {c1}  <->  {c2}  :  r = {v}')
else:
    print('No high-correlation pairs above 0.85.')

---
## 10. Churn Timing — When Did Members Go Silent?

In [ ]:
last_flight = (
    activity[activity[FLIGHTS_COL]>0]
    .groupby(KEY)['period'].max().reset_index()
    .rename(columns={'period':'last_flight_month'})
)
last_flight = last_flight.merge(labels, on=KEY, how='inner')
churned_silence = last_flight[last_flight['churned']==1].copy()

silence_by_month = (
    churned_silence.groupby('last_flight_month').size()
    .reset_index(name='n_went_silent')
    .sort_values('last_flight_month')
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(silence_by_month['last_flight_month'].astype(str),
       silence_by_month['n_went_silent'],
       color=CHURN_COLOR, alpha=0.75, edgecolor='white', width=0.8)
ax.set_xlabel('Last flight month')
ax.set_ylabel('Members who went silent')
ax.set_title('When churners took their last flight — monthly distribution')
ticks = ax.get_xticklabels()
for i, tick in enumerate(ticks):
    if i % 6 != 0: tick.set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG / '03_churn_timing.png', bbox_inches='tight')
plt.show()

---
## 11. EDA Summary — Key Numbers for Your Report

In [ ]:
overall_churn_rate  = members['churned'].mean()
high_clv_churn_rate = members[members[CLV_COL]>=members[CLV_COL].quantile(0.75)]['churned'].mean()
median_red_c        = members[members['churned']==1]['redemption_ratio'].median()
median_red_r        = members[members['churned']==0]['redemption_ratio'].median()

print('='*60)
print('  EDA KEY FINDINGS — copy these into your report')
print('='*60)
print(f'  1. Overall churn rate                  : {overall_churn_rate:.1%}')
print(f'  2. Top-25% CLV member churn rate        : {high_clv_churn_rate:.1%}')
print(f'     -> CLV does not protect against churn')
print(f'  3. Churner median redemption ratio      : {median_red_c:.3f}')
print(f'     Retained median redemption ratio     : {median_red_r:.3f}')
print(f'     -> Loss aversion signal confirmed')
print(f'  4. Declining trajectory churn rate      : {declining_churn:.1%}')
print(f'     Stable trajectory churn rate         : {stable_churn:.1%}')
print(f'     -> Recency matters more than history')
print('='*60)

members.to_csv(PROC / '03_members_eda.csv', index=False)
print('\nSaved: 03_members_eda.csv')
print('Next step -> 04_feature_engineering.ipynb')